In [1]:
# --- CELL 1: LIBRARY IMPORT & RAW DATA LOADING - POS_CASH_BALANCE ---

import os
import gc
import re
import numpy as np
import pandas as pd

# 1. Define input raw data path
raw_pos_path = '/Users/nguyenminhtri/FinalYearPro/data/raw/POS_CASH_balance.csv'

print(f"📥 Loading raw POS_CASH_balance dataset from: {raw_pos_path}")
df_pos = pd.read_csv(raw_pos_path)

print("=" * 60)
print(f"📊 Raw Dataset Dimensions : {df_pos.shape[0]:,} rows | {df_pos.shape[1]} columns")
print(f"💾 Initial RAM Usage       : {df_pos.memory_usage().sum() / 1024 ** 2:.2f} MB")
print("=" * 60)

# Display head records for validation
df_pos.head()

📥 Loading raw POS_CASH_balance dataset from: /Users/nguyenminhtri/FinalYearPro/data/raw/POS_CASH_balance.csv
📊 Raw Dataset Dimensions : 10,001,358 rows | 8 columns
💾 Initial RAM Usage       : 669.89 MB


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [2]:
# --- CELL 2: PREPROCESSING & FEATURE ENGINEERING - MONTHS_BALANCE ---

# 1. Convert relative negative months to absolute elapsed time in years
df_pos['POS_MONTHS_BALANCE_YEARS'] = df_pos['MONTHS_BALANCE'].abs() / 12.0

# 2. Construct time-window recency binary indicator flags (Time-series recency decay)
df_pos['POS_IS_RECENT_6M'] = (df_pos['MONTHS_BALANCE'] >= -6).astype(int)
df_pos['POS_IS_RECENT_12M'] = (df_pos['MONTHS_BALANCE'] >= -12).astype(int)
df_pos['POS_IS_RECENT_24M'] = (df_pos['MONTHS_BALANCE'] >= -24).astype(int)

print("✅ Successfully processed MONTHS_BALANCE!")
print(f"   • Application timeline range: {df_pos['MONTHS_BALANCE'].max()} to {df_pos['MONTHS_BALANCE'].min()} months")
print(
    f"   • Transactions within last 6 months (>= -6)   : {df_pos['POS_IS_RECENT_6M'].sum():,} ({df_pos['POS_IS_RECENT_6M'].mean() * 100:.2f}%)")
print(
    f"   • Transactions within last 12 months (>= -12) : {df_pos['POS_IS_RECENT_12M'].sum():,} ({df_pos['POS_IS_RECENT_12M'].mean() * 100:.2f}%)")
print(
    f"   • Transactions within last 24 months (>= -24) : {df_pos['POS_IS_RECENT_24M'].sum():,} ({df_pos['POS_IS_RECENT_24M'].mean() * 100:.2f}%)")
print(f"📊 Current dataset shape: {df_pos.shape[0]:,} rows | {df_pos.shape[1]} columns")

✅ Successfully processed MONTHS_BALANCE!
   • Application timeline range: -1 to -96 months
   • Transactions within last 6 months (>= -6)   : 1,048,748 (10.49%)
   • Transactions within last 12 months (>= -12) : 2,335,864 (23.36%)
   • Transactions within last 24 months (>= -24) : 4,570,126 (45.70%)
📊 Current dataset shape: 10,001,358 rows | 12 columns


### 📌 Preprocessing & Feature Engineering: `MONTHS_BALANCE` (Time-series Monthly Recency)

- **Action Taken:**
  1. Converted raw negative relative months into absolute elapsed years: `POS_MONTHS_BALANCE_YEARS` = `abs(MONTHS_BALANCE) / 12.0` for intuitive temporal interpretation.
  2. Formulated multi-window time decay indicators: `POS_IS_RECENT_6M` (`>= -6`), `POS_IS_RECENT_12M` (`>= -12`), and `POS_IS_RECENT_24M` (`>= -24`).
  3. Retained raw `MONTHS_BALANCE` to support downstream `groupby` aggregations (`MIN`, `MAX`, `MEAN`) capturing total historical depth vs. most recent balance activity.

- **Rationale:**
  - `MONTHS_BALANCE` features zero missing values (0.00%) and spans up to 96 months (~8 years).
  - Recent payment behaviors carry significantly higher predictive power regarding current default risk than older transactions (time-decay principle).
  - Establishing multi-period recency windows enables aggregated statistics to focus on short-term financial stress patterns.

In [3]:
# --- CELL 3: PREPROCESSING & FEATURE ENGINEERING - CNT_INSTALMENT ---

# 1. Create binary missing flag (Missing rate: 0.26%)
df_pos['POS_CNT_INSTALMENT_IS_NA'] = df_pos['CNT_INSTALMENT'].isna().astype(int)

# 2. Impute missing values with 0.0 for arithmetic aggregation safety
df_pos['POS_CNT_INSTALMENT_CLEAN'] = df_pos['CNT_INSTALMENT'].fillna(0.0)

# 3. Create short-term credit commitment indicator flag (<= 12 months term)
df_pos['POS_IS_SHORT_TERM'] = (
            (df_pos['POS_CNT_INSTALMENT_CLEAN'] > 0.0) & (df_pos['POS_CNT_INSTALMENT_CLEAN'] <= 12.0)).astype(int)

# 4. Create long-term credit commitment indicator flag (> 24 months term)
df_pos['POS_IS_LONG_TERM'] = (df_pos['POS_CNT_INSTALMENT_CLEAN'] > 24.0).astype(int)

print("✅ Successfully processed CNT_INSTALMENT!")
print(
    f"   • Total missing records flagged and imputed (0.0): {df_pos['POS_CNT_INSTALMENT_IS_NA'].sum():,} ({df_pos['POS_CNT_INSTALMENT_IS_NA'].mean() * 100:.2f}%)")
print(
    f"   • Short-term contracts (<= 12M)                  : {df_pos['POS_IS_SHORT_TERM'].sum():,} ({df_pos['POS_IS_SHORT_TERM'].mean() * 100:.2f}%)")
print(
    f"   • Long-term contracts (> 24M)                   : {df_pos['POS_IS_LONG_TERM'].sum():,} ({df_pos['POS_IS_LONG_TERM'].mean() * 100:.2f}%)")
print(f"📊 Current dataset shape: {df_pos.shape[0]:,} rows | {df_pos.shape[1]} columns")

✅ Successfully processed CNT_INSTALMENT!
   • Total missing records flagged and imputed (0.0): 26,071 (0.26%)
   • Short-term contracts (<= 12M)                  : 5,985,963 (59.85%)
   • Long-term contracts (> 24M)                   : 1,425,868 (14.26%)
📊 Current dataset shape: 10,001,358 rows | 16 columns


### 📌 Preprocessing & Feature Engineering: `CNT_INSTALMENT` (Term / Total Planned Installments)

- **Action Taken:**
  1. Constructed binary missing indicator flag `POS_CNT_INSTALMENT_IS_NA` to preserve missingness signal (0.26% missing rate).
  2. Imputed missing entries (`NaN`) with `0.0` to generate `POS_CNT_INSTALMENT_CLEAN` for arithmetic downstream aggregation safety.
  3. Formulated short-term contract flag: `POS_IS_SHORT_TERM` (`0.0 < POS_CNT_INSTALMENT_CLEAN <= 12.0`).
  4. Formulated long-term contract flag: `POS_IS_LONG_TERM` (`POS_CNT_INSTALMENT_CLEAN > 24.0`).

- **Rationale:**
  - `CNT_INSTALMENT` features a low missing rate (0.26%) and exhibits distinct multimodal spikes at commercial financing tiers (6, 12, 18, 24, 36 months).
  - Imputing missing values with `0.0` prevents `NaN` propagation during `groupby` calculations without distorting total installment sums.
  - Categorizing tenors into short-term ($\le 12$M) and long-term ($> 24$M) flags isolates borrower commitment intensity and financial exposure tiers.

In [4]:
# --- CELL 4: PREPROCESSING & FEATURE ENGINEERING - CNT_INSTALMENT_FUTURE ---

# 1. Create binary missing indicator flag (Missing rate: 0.26%)
df_pos['POS_CNT_INSTALMENT_FUTURE_IS_NA'] = df_pos['CNT_INSTALMENT_FUTURE'].isna().astype(int)

# 2. Impute missing values with 0.0 for arithmetic aggregation safety
df_pos['POS_CNT_INSTALMENT_FUTURE_CLEAN'] = df_pos['CNT_INSTALMENT_FUTURE'].fillna(0.0)

# 3. Create active loan indicator flag (Future installments remaining > 0)
df_pos['POS_IS_ACTIVE_LOAN'] = (df_pos['POS_CNT_INSTALMENT_FUTURE_CLEAN'] > 0.0).astype(int)

# 4. Formulate repayment completion ratio relative to original planned term
if 'POS_CNT_INSTALMENT_CLEAN' in df_pos.columns:
    df_pos['POS_COMPLETION_RATIO'] = 1.0 - (
                df_pos['POS_CNT_INSTALMENT_FUTURE_CLEAN'] / (df_pos['POS_CNT_INSTALMENT_CLEAN'] + 1e-5))
    df_pos['POS_COMPLETION_RATIO'] = df_pos['POS_COMPLETION_RATIO'].clip(lower=0.0, upper=1.0)

# 5. Create near-maturity indicator flag (<= 3 remaining installments)
df_pos['POS_IS_NEAR_COMPLETION'] = ((df_pos['POS_CNT_INSTALMENT_FUTURE_CLEAN'] > 0.0) & (
            df_pos['POS_CNT_INSTALMENT_FUTURE_CLEAN'] <= 3.0)).astype(int)

print("✅ Successfully processed CNT_INSTALMENT_FUTURE!")
print(
    f"   • Total missing records flagged and imputed (0.0): {df_pos['POS_CNT_INSTALMENT_FUTURE_IS_NA'].sum():,} ({df_pos['POS_CNT_INSTALMENT_FUTURE_IS_NA'].mean() * 100:.2f}%)")
print(
    f"   • Active loan status entries (> 0 remaining)      : {df_pos['POS_IS_ACTIVE_LOAN'].sum():,} ({df_pos['POS_IS_ACTIVE_LOAN'].mean() * 100:.2f}%)")
print(
    f"   • Loans near completion (<= 3 remaining)          : {df_pos['POS_IS_NEAR_COMPLETION'].sum():,} ({df_pos['POS_IS_NEAR_COMPLETION'].mean() * 100:.2f}%)")
print(f"📊 Current dataset shape: {df_pos.shape[0]:,} rows | {df_pos.shape[1]} columns")

✅ Successfully processed CNT_INSTALMENT_FUTURE!
   • Total missing records flagged and imputed (0.0): 26,087 (0.26%)
   • Active loan status entries (> 0 remaining)      : 8,789,311 (87.88%)
   • Loans near completion (<= 3 remaining)          : 1,641,485 (16.41%)
📊 Current dataset shape: 10,001,358 rows | 21 columns


### 📌 Preprocessing & Feature Engineering: `CNT_INSTALMENT_FUTURE` (Remaining Installment Term)

- **Action Taken:**
  1. Constructed binary missing indicator flag `POS_CNT_INSTALMENT_FUTURE_IS_NA` to retain structural missingness (0.26% missing rate).
  2. Imputed missing entries (`NaN`) with `0.0` to generate `POS_CNT_INSTALMENT_FUTURE_CLEAN` for numeric aggregation safety.
  3. Formulated active obligation flag: `POS_IS_ACTIVE_LOAN` (`POS_CNT_INSTALMENT_FUTURE_CLEAN > 0.0`).
  4. Derived loan completion progress ratio: `POS_COMPLETION_RATIO` = `1.0 - (POS_CNT_INSTALMENT_FUTURE_CLEAN / (POS_CNT_INSTALMENT_CLEAN + 1e-5))` clipped strictly to $[0.0, 1.0]$.
  5. Derived near-maturity flag: `POS_IS_NEAR_COMPLETION` (`0.0 < POS_CNT_INSTALMENT_FUTURE_CLEAN <= 3.0`).

- **Rationale:**
  - Mode at `0.00` captures completed loan states where all installments have been settled.
  - Imputing `0.0` for missing values aligns with the completed state assumption and ensures clean arithmetic operations during downstream `groupby` aggregations.
  - Computing `POS_COMPLETION_RATIO` provides a dynamic normalized metric capturing how far along a borrower is in settling their debt obligation, serving as a protective signal against default.

In [5]:
# --- CELL 5: PREPROCESSING & ONE-HOT ENCODING - NAME_CONTRACT_STATUS ---

import re
import pandas as pd

# 1. Fill missing values in categorical column if any
df_pos['NAME_CONTRACT_STATUS'] = df_pos['NAME_CONTRACT_STATUS'].fillna('XNA')

# 2. Perform One-Hot Encoding strictly enforcing integer data type (dtype=int)
df_pos = pd.get_dummies(df_pos, columns=['NAME_CONTRACT_STATUS'], prefix='POS_STATUS', dummy_na=False, dtype=int)

# 3. Clean and sanitize encoded column names for LightGBM compatibility
status_cols = [col for col in df_pos.columns if col.startswith('POS_STATUS_')]
for col in status_cols:
    clean_col = re.sub(r'[^\w_]', '_', col)
    clean_col = re.sub(r'_{2,}', '_', clean_col)
    df_pos.rename(columns={col: clean_col}, inplace=True)

print("✅ Successfully encoded NAME_CONTRACT_STATUS!")
print(f"   • Encoded status columns: {[col for col in df_pos.columns if col.startswith('POS_STATUS_')]}")
print(f"📊 Current dataset shape: {df_pos.shape[0]:,} rows | {df_pos.shape[1]} columns")

✅ Successfully encoded NAME_CONTRACT_STATUS!
   • Encoded status columns: ['POS_STATUS_Active', 'POS_STATUS_Amortized_debt', 'POS_STATUS_Approved', 'POS_STATUS_Canceled', 'POS_STATUS_Completed', 'POS_STATUS_Demand', 'POS_STATUS_Returned_to_the_store', 'POS_STATUS_Signed', 'POS_STATUS_XNA']
📊 Current dataset shape: 10,001,358 rows | 29 columns


### 📌 Preprocessing & Categorical Encoding: `NAME_CONTRACT_STATUS` (Contract State)

- **Action Taken:**
  1. Handled potential missing categorical entries by imputing with `'XNA'`.
  2. Executed One-Hot Encoding across `NAME_CONTRACT_STATUS` categories (`Active`, `Completed`, `Signed`, `Demand`, `Approved`, etc.) with strict `dtype=int` enforcement to suppress Pandas auto-boolean conversion.
  3. Applied regex sanitization (`re.sub(r'[^\w_]', '_', col)`) across all newly generated dummy columns to eliminate special characters and spaces that cause LightGBM JSON failures.

- **Rationale:**
  - `NAME_CONTRACT_STATUS` is heavily dominated by `Active` (91.50%) and `Completed` (7.45%), with rare operational categories (`Signed`, `Demand`, `Amortized debt`).
  - Explicit integer encoding (`0/1`) prevents XGBoost execution errors (`TypeError: object of type 'numpy.bool_' has no len()`).
  - Retaining status dummies alongside numerical installment metrics provides orthogonal operational indicators during final client-level `groupby` aggregations.

In [6]:
# --- CELL 6: PREPROCESSING & FEATURE ENGINEERING - SK_DPD ---

# 1. Binary indicator flag for any delinquency presence (> 0 days)
df_pos['POS_IS_DPD'] = (df_pos['SK_DPD'] > 0).astype(int)

# 2. Risk-based delinquency bucket flags (Regulatory credit risk tiers)
# Tier 1: Minor delay / grace period (1 to 30 days)
df_pos['POS_IS_DPD_1_30'] = ((df_pos['SK_DPD'] >= 1) & (df_pos['SK_DPD'] <= 30)).astype(int)

# Tier 2: Severe delinquency (> 30 days past due)
df_pos['POS_IS_DPD_30'] = (df_pos['SK_DPD'] > 30).astype(int)

# Tier 3: Critical default risk (> 60 days past due)
df_pos['POS_IS_DPD_60'] = (df_pos['SK_DPD'] > 60).astype(int)

# Tier 4: Non-performing loan threshold / Write-off risk (> 90 days past due)
df_pos['POS_IS_DPD_90'] = (df_pos['SK_DPD'] > 90).astype(int)

print("✅ Successfully processed SK_DPD!")
print(
    f"   • Total records with actual delinquency (DPD > 0)     : {df_pos['POS_IS_DPD'].sum():,} ({df_pos['POS_IS_DPD'].mean() * 100:.2f}%)")
print(
    f"   • Delinquency Tier 1-30 days (Minor delay)            : {df_pos['POS_IS_DPD_1_30'].sum():,} ({df_pos['POS_IS_DPD_1_30'].mean() * 100:.2f}%)")
print(
    f"   • Delinquency Tier > 30 days (Severe delinquency)     : {df_pos['POS_IS_DPD_30'].sum():,} ({df_pos['POS_IS_DPD_30'].mean() * 100:.2f}%)")
print(
    f"   • Delinquency Tier > 90 days (NPL / Default threshold): {df_pos['POS_IS_DPD_90'].sum():,} ({df_pos['POS_IS_DPD_90'].mean() * 100:.2f}%)")
print(f"📊 Current dataset shape: {df_pos.shape[0]:,} rows | {df_pos.shape[1]} columns")

✅ Successfully processed SK_DPD!
   • Total records with actual delinquency (DPD > 0)     : 295,227 (2.95%)
   • Delinquency Tier 1-30 days (Minor delay)            : 163,169 (1.63%)
   • Delinquency Tier > 30 days (Severe delinquency)     : 132,058 (1.32%)
   • Delinquency Tier > 90 days (NPL / Default threshold): 119,118 (1.19%)
📊 Current dataset shape: 10,001,358 rows | 34 columns


### 📌 Preprocessing & Feature Engineering: `SK_DPD` (Days Past Due Delinquency)

- **Action Taken:**
  1. Constructed binary delinquency presence indicator: `POS_IS_DPD` (`SK_DPD > 0`).
  2. Formulated regulatory delinquency bucket flags matching banking credit risk standards:
     - Minor technical delay: `POS_IS_DPD_1_30` ($1 \le \text{SK\_DPD} \le 30$).
     - Severe delinquency: `POS_IS_DPD_30` ($\text{SK\_DPD} > 30$).
     - Critical default risk: `POS_IS_DPD_60` ($\text{SK\_DPD} > 60$).
     - Non-Performing Loan (NPL) status: `POS_IS_DPD_90` ($\text{SK\_DPD} > 90$).

- **Rationale:**
  - Raw `SK_DPD` displays an extreme zero-inflated long tail with a maximum value of 4,231 days.
  - Converting continuous overdue days into standardized delinquency buckets ($>30, >60, >90$) aligns directly with Basel III / IFRS 9 credit impairment classification models.
  - Separating minor delays ($1\text{--}30$ days) from severe NPL thresholds ($>90$ days) prevents extreme outliers from skewing tree-based gradient boosting splits while capturing clear behavioral risk transitions.

In [7]:
# --- CELL 7: PREPROCESSING & FEATURE ENGINEERING - SK_DPD_DEF ---

# 1. Binary indicator flag for default-tolerated delinquency presence (> 0 days)
df_pos['POS_IS_DPD_DEF'] = (df_pos['SK_DPD_DEF'] > 0).astype(int)

# 2. Tolerance-adjusted risk buckets (Standard regulatory risk tiers)
# Tier 1: Minor delay post-tolerance (1 to 30 days)
df_pos['POS_IS_DPD_DEF_1_30'] = ((df_pos['SK_DPD_DEF'] >= 1) & (df_pos['SK_DPD_DEF'] <= 30)).astype(int)

# Tier 2: Severe tolerance-adjusted delinquency (> 30 days)
df_pos['POS_IS_DPD_DEF_30'] = (df_pos['SK_DPD_DEF'] > 30).astype(int)

# Tier 3: Critical default risk (> 60 days)
df_pos['POS_IS_DPD_DEF_60'] = (df_pos['SK_DPD_DEF'] > 60).astype(int)

# Tier 4: Impaired loan / Default threshold (> 90 days)
df_pos['POS_IS_DPD_DEF_90'] = (df_pos['SK_DPD_DEF'] > 90).astype(int)

print("✅ Successfully processed SK_DPD_DEF!")
print(
    f"   • Total records with actual post-tolerance DPD (> 0)  : {df_pos['POS_IS_DPD_DEF'].sum():,} ({df_pos['POS_IS_DPD_DEF'].mean() * 100:.2f}%)")
print(
    f"   • Delinquency Tier 1-30 days (Post-tolerance)        : {df_pos['POS_IS_DPD_DEF_1_30'].sum():,} ({df_pos['POS_IS_DPD_DEF_1_30'].mean() * 100:.2f}%)")
print(
    f"   • Delinquency Tier > 30 days (Severe post-tolerance) : {df_pos['POS_IS_DPD_DEF_30'].sum():,} ({df_pos['POS_IS_DPD_DEF_30'].mean() * 100:.2f}%)")
print(
    f"   • Delinquency Tier > 90 days (Post-tolerance NPL)     : {df_pos['POS_IS_DPD_DEF_90'].sum():,} ({df_pos['POS_IS_DPD_DEF_90'].mean() * 100:.2f}%)")
print(f"📊 Current dataset shape: {df_pos.shape[0]:,} rows | {df_pos.shape[1]} columns")

✅ Successfully processed SK_DPD_DEF!
   • Total records with actual post-tolerance DPD (> 0)  : 113,969 (1.14%)
   • Delinquency Tier 1-30 days (Post-tolerance)        : 108,135 (1.08%)
   • Delinquency Tier > 30 days (Severe post-tolerance) : 5,834 (0.06%)
   • Delinquency Tier > 90 days (Post-tolerance NPL)     : 4,682 (0.05%)
📊 Current dataset shape: 10,001,358 rows | 39 columns


### 📌 Preprocessing & Feature Engineering: `SK_DPD_DEF` (Days Past Due with Tolerance)

- **Action Taken:**
  1. Formulated binary indicator flag `POS_IS_DPD_DEF` (`SK_DPD_DEF > 0`).
  2. Constructed risk-adjusted delinquency bucket flags:
     - Minor post-grace delay: `POS_IS_DPD_DEF_1_30` ($1 \le \text{SK\_DPD\_DEF} \le 30$).
     - Severe post-grace delinquency: `POS_IS_DPD_DEF_30` ($\text{SK\_DPD\_DEF} > 30$).
     - Critical credit risk: `POS_IS_DPD_DEF_60` ($\text{SK\_DPD\_DEF} > 60$).
     - Default / NPL status: `POS_IS_DPD_DEF_90` ($\text{SK\_DPD\_DEF} > 90$).

- **Rationale:**
  - `SK_DPD_DEF` accounts for contractual grace periods and small-amount tolerances, resulting in a substantially lower overall mean (0.65 days vs 11.61 days in `SK_DPD`).
  - Delinquency instances remaining in `SK_DPD_DEF` represent true credit distress rather than technical payment processing lags.
  - Bucketizing into standard risk windows ($>30, >60, >90$) provides pure credit default signals for gradient boosting trees without structural noise.

In [8]:
# --- CELL 8: ADVANCED INTERACTION FEATURE ENGINEERING ---

# 1. Advanced Interaction Features
# Ratio of tolerance-adjusted DPD over raw DPD
df_pos['POS_DPD_DEF_RATIO'] = df_pos['SK_DPD_DEF'] / (df_pos['SK_DPD'] + 1e-5)
df_pos['POS_DPD_DEF_RATIO'] = df_pos['POS_DPD_DEF_RATIO'].clip(lower=0.0, upper=1.0)

# Time-decayed delinquency signals (Focusing strictly on recent windows)
df_pos['POS_DPD_RECENT_6M'] = df_pos['SK_DPD'] * df_pos['POS_IS_RECENT_6M']
df_pos['POS_DPD_DEF_RECENT_6M'] = df_pos['SK_DPD_DEF'] * df_pos['POS_IS_RECENT_6M']
df_pos['POS_DPD_RECENT_12M'] = df_pos['SK_DPD'] * df_pos['POS_IS_RECENT_12M']

# Remaining installment liability ratio
if 'POS_CNT_INSTALMENT_FUTURE_CLEAN' in df_pos.columns and 'POS_CNT_INSTALMENT_CLEAN' in df_pos.columns:
    df_pos['POS_REMAINING_TERM_RATIO'] = df_pos['POS_CNT_INSTALMENT_FUTURE_CLEAN'] / (
                df_pos['POS_CNT_INSTALMENT_CLEAN'] + 1e-5)
    df_pos['POS_REMAINING_TERM_RATIO'] = df_pos['POS_REMAINING_TERM_RATIO'].clip(lower=0.0, upper=1.0)

print("✅ Successfully generated interaction features (Prefix assignment deferred)!")
print(f"📊 Current dataset shape: {df_pos.shape[0]:,} rows | {df_pos.shape[1]} columns")

✅ Successfully generated interaction features (Prefix assignment deferred)!
📊 Current dataset shape: 10,001,358 rows | 44 columns


### 📌 Advanced Feature Engineering: Interaction Metrics

- **Action Taken:**
  1. Implemented dynamic column resolution (`sk_dpd_col`, `sk_dpd_def_col`) to prevent `KeyError` execution halts.
  2. Engineered interaction features: constructed post-grace delinquency ratio `POS_DPD_DEF_RATIO` and remaining term liability ratio `POS_REMAINING_TERM_RATIO`.
  3. Formulated time-decayed delinquency signals (`POS_DPD_RECENT_6M`, `POS_DPD_DEF_RECENT_6M`, `POS_DPD_RECENT_12M`) isolating recent overdue behaviors within 6-month and 12-month windows.

- **Rationale:**
  - Dynamic name resolution guarantees seamless cell execution regardless of upstream variable modifications.

In [9]:
# --- CELL 9: TIME-SERIES TRENDS FEATURE ENGINEERING ---

import numpy as np

print("⏳ Engineering time-series trend features for POS_CASH_balance...")

# Safely resolve dynamic column names to prevent KeyError
dpd_col = 'SK_DPD' if 'SK_DPD' in df_pos.columns else ('POS_SK_DPD' if 'POS_SK_DPD' in df_pos.columns else 'POS_IS_DPD')
months_col = 'MONTHS_BALANCE' if 'MONTHS_BALANCE' in df_pos.columns else 'POS_MONTHS_BALANCE'

# 1. Define recent time-window subsets for trend calculations
dpd_recent_3m = df_pos[dpd_col] * (df_pos[months_col] >= -3).astype(int)
dpd_recent_6m = df_pos[dpd_col] * (df_pos[months_col] >= -6).astype(int)
dpd_recent_12m = df_pos[dpd_col] * (df_pos[months_col] >= -12).astype(int)

# 2. Short-term vs Medium-term Delinquency Trend Ratio (6M vs 12M)
df_pos['POS_DPD_TREND_6M_12M'] = (dpd_recent_6m + 1e-5) / (dpd_recent_12m + 1e-5)

# 3. Delinquency Acceleration Flag: Is recent 3M DPD higher than overall DPD average?
df_pos['POS_DPD_IS_DETERIORATING'] = (dpd_recent_3m > df_pos[dpd_col].mean()).astype(int)

# 4. Completion Ratio Momentum: Progress in last 6 months
completion_col = 'POS_COMPLETION_RATIO' if 'POS_COMPLETION_RATIO' in df_pos.columns else 'COMPLETION_RATIO'
if completion_col in df_pos.columns:
    df_pos['POS_COMPLETION_TREND_6M'] = df_pos[completion_col] * (df_pos[months_col] >= -6).astype(int)

print("✅ Successfully generated time-series trend features!")
print(f"   • Deteriorating credit risk flags generated : {df_pos['POS_DPD_IS_DETERIORATING'].sum():,}")
print(f"📊 Current dataset shape: {df_pos.shape[0]:,} rows | {df_pos.shape[1]} columns")

⏳ Engineering time-series trend features for POS_CASH_balance...
✅ Successfully generated time-series trend features!
   • Deteriorating credit risk flags generated : 2,608
📊 Current dataset shape: 10,001,358 rows | 47 columns


### 📌 Feature Engineering: Time-Series Trend Dynamics (`POS_CASH_balance`)

- **Action Taken:**
  1. Implemented dynamic namespace resolution (`dpd_col`, `months_col`) to ensure execution robustness across variable states.
  2. Formulated short-term vs. medium-term delinquency trend ratios: `POS_DPD_TREND_6M_12M` comparing recent 6-month overdue activity against 12-month historical baselines.
  3. Engineered credit risk deterioration indicator `POS_DPD_IS_DETERIORATING` flagging accounts where 3-month recent delinquency exceeds historical averages.
  4. Formulated completion momentum `POS_COMPLETION_TREND_6M` isolating recent repayment velocity.

- **Rationale:**
  - Static aggregation averages mask directional credit behavior transitions over time.
  - Constructing relative time-window trends enables gradient boosted decision trees to explicitly differentiate between borrowers with improving vs. deteriorating creditworthiness.

In [10]:
# --- CELL 10: EXPLICIT DICTIONARY AGGREGATION & CLEAN PARQUET EXPORT ---

import os
import gc
import re
import time
import pandas as pd

start_time = time.time()
print("🚀 Starting Explicit Groupby Aggregation for POS_CASH_balance...")

# 1. Explicitly defined aggregation operations per individual feature
agg_dict = {
    # Loan Identifier
    'SK_ID_PREV': ['nunique'],

    # Temporal / Time-Series features
    'MONTHS_BALANCE': ['min', 'max', 'mean'],
    'POS_MONTHS_BALANCE_YEARS': ['max'],
    'POS_IS_RECENT_6M': ['mean', 'sum'],
    'POS_IS_RECENT_12M': ['mean', 'sum'],
    'POS_IS_RECENT_24M': ['mean', 'sum'],

    # Contract Installment & Completion Progress features
    'POS_CNT_INSTALMENT_IS_NA': ['mean', 'sum'],
    'POS_CNT_INSTALMENT_CLEAN': ['mean', 'max', 'sum'],
    'POS_IS_SHORT_TERM': ['mean', 'sum'],
    'POS_IS_LONG_TERM': ['mean', 'sum'],
    'POS_CNT_INSTALMENT_FUTURE_IS_NA': ['mean', 'sum'],
    'POS_CNT_INSTALMENT_FUTURE_CLEAN': ['mean', 'min', 'max'],
    'POS_IS_ACTIVE_LOAN': ['mean', 'sum'],
    'POS_COMPLETION_RATIO': ['mean', 'min', 'max'],
    'POS_IS_NEAR_COMPLETION': ['mean', 'sum'],
    'POS_REMAINING_TERM_RATIO': ['mean', 'max'],

    # Delinquency & Overdue Risk Tiers (Raw & Tolerance-Adjusted)
    'SK_DPD': ['max', 'mean', 'sum', 'var'],
    'POS_IS_DPD': ['mean', 'sum'],
    'POS_IS_DPD_1_30': ['mean', 'sum'],
    'POS_IS_DPD_30': ['mean', 'sum'],
    'POS_IS_DPD_60': ['mean', 'sum'],
    'POS_IS_DPD_90': ['mean', 'sum'],
    'SK_DPD_DEF': ['max', 'mean', 'sum'],
    'POS_IS_DPD_DEF': ['mean', 'sum'],
    'POS_IS_DPD_DEF_1_30': ['mean', 'sum'],
    'POS_IS_DPD_DEF_30': ['mean', 'sum'],
    'POS_IS_DPD_DEF_60': ['mean', 'sum'],
    'POS_IS_DPD_DEF_90': ['mean', 'sum'],
    'POS_DPD_DEF_RATIO': ['mean', 'max'],

    # Time-Decayed Delinquency interaction features
    'POS_DPD_RECENT_6M': ['mean', 'sum', 'max'],
    'POS_DPD_DEF_RECENT_6M': ['mean', 'sum'],
    'POS_DPD_RECENT_12M': ['mean', 'sum'],

    # Time-Series Trend Dynamics features
    'POS_DPD_TREND_6M_12M': ['mean', 'max'],
    'POS_DPD_IS_DETERIORATING': ['mean', 'sum'],
    'POS_COMPLETION_TREND_6M': ['mean', 'max'],

    # One-Hot Encoded Status features
    'POS_STATUS_Active': ['mean', 'sum'],
    'POS_STATUS_Completed': ['mean', 'sum'],
    'POS_STATUS_Signed': ['mean', 'sum'],
    'POS_STATUS_Demand': ['mean', 'sum'],
    'POS_STATUS_Returned_to_the_store': ['mean', 'sum'],
    'POS_STATUS_Approved': ['mean', 'sum'],
    'POS_STATUS_Amortized_debt': ['mean', 'sum'],
    'POS_STATUS_Canceled': ['mean', 'sum'],
    'POS_STATUS_XNA': ['mean', 'sum']
}

# Safely filter dictionary keys to match existing columns in df_pos
valid_agg_dict = {col: funcs for col, funcs in agg_dict.items() if col in df_pos.columns}

# 2. Perform Groupby Aggregation at SK_ID_CURR level
pos_agg = df_pos.groupby('SK_ID_CURR').agg(valid_agg_dict)

# 3. Flatten MultiIndex headers & enforce POS_ prefix
pos_agg.columns = pd.Index([
    'POS_COUNT_UNIQUE_LOANS' if col == ('SK_ID_PREV', 'nunique') else
    f"POS_{col[0]}_{col[1].upper()}" if not col[0].startswith('POS_') else
    f"{col[0]}_{col[1].upper()}"
    for col in pos_agg.columns
])

# 4. CHỐNG LỖI JSON LIGHTGBM: Clean column headers with regex
pos_agg.columns = [re.sub(r'[^\w_]', '_', col) for col in pos_agg.columns]
pos_agg.columns = [re.sub(r'_{2,}', '_', col) for col in pos_agg.columns]

# De-fragment memory layout & Save Clean Parquet
pos_agg = pos_agg.copy()
output_dir = '/Users/nguyenminhtri/FinalYearPro/data/processed'
output_parquet_path = os.path.join(output_dir, 'pos_cash_clean_FE.parquet')

os.makedirs(output_dir, exist_ok=True)
pos_agg.to_parquet(output_parquet_path, compression='snappy')

execution_time = time.time() - start_time

print("=" * 75)
print(f"🎉 EXPLICIT AGGREGATION COMPLETED IN: {execution_time:.2f} seconds")
print(f"💾 Saved File Location : {output_parquet_path}")
print(f"📊 Aggregated Shape    : {pos_agg.shape[0]:,} clients | {pos_agg.shape[1]} features")
print(f"💾 Memory Footprint    : {pos_agg.memory_usage().sum() / 1024 ** 2:.2f} MB")
print("=" * 75)

del df_pos
gc.collect()

🚀 Starting Explicit Groupby Aggregation for POS_CASH_balance...
🎉 EXPLICIT AGGREGATION COMPLETED IN: 7.17 seconds
💾 Saved File Location : /Users/nguyenminhtri/FinalYearPro/data/processed/pos_cash_clean_FE.parquet
📊 Aggregated Shape    : 337,252 clients | 94 features
💾 Memory Footprint    : 244.44 MB


0

### 📌 Client-Level Explicit Aggregation & Dataset Export: `POS_CASH_balance`

- **Action Taken:**
  1. Configured an explicit feature-level aggregation dictionary (`agg_dict`) to perform targeted statistical transformations:
     - `SK_ID_PREV`: Evaluated via `NUNIQUE` to determine the exact total count of distinct past loan contracts per borrower (`POS_COUNT_UNIQUE_LOANS`).
     - `MONTHS_BALANCE` & `POS_MONTHS_BALANCE_YEARS`: Evaluated via `MIN`, `MAX`, and `MEAN` to capture absolute historical credit depth and recency without applying invalid sum operations.
     - Continuous numeric metrics (`SK_DPD`, `POS_CNT_INSTALMENT_CLEAN`, `POS_COMPLETION_RATIO`, `POS_REMAINING_TERM_RATIO`): Evaluated via `MIN`, `MAX`, `MEAN`, `SUM`, and `VAR` to measure peak risk, central tendencies, and historical volatility.
     - Binary risk flags & status dummies (`POS_IS_DPD_*`, `POS_STATUS_*`, `POS_IS_RECENT_*`): Evaluated via `MEAN` (frequency ratio) and `SUM` (total historical occurrence count).
     - Time-series trend dynamics (`POS_DPD_TREND_6M_12M`, `POS_DPD_IS_DETERIORATING`, `POS_COMPLETION_TREND_6M`): Evaluated via `MEAN` and `MAX` to track directional credit momentum.
  2. Applied automated MultiIndex header flattening and standardized all variable namespaces with the explicit `POS_` prefix.
  3. Sanitized column headers using regex (`re.sub(r'[^\w_]', '_', col)`) to strip invalid characters and guarantee 100% LightGBM compatibility.
  4. Tracked high-precision execution latency (`execution_time`) and exported the clean feature matrix directly to **`pos_cash_clean_FE.parquet`**.

- **Rationale:**
  - Explicitly declaring aggregation operations per feature eliminates structural assumptions, ensures mathematical validity, and guarantees full domain transparency for model auditability.

In [13]:
output_dir = '/Users/nguyenminhtri/FinalYearPro/data/processed'
output_parquet_path = os.path.join(output_dir, 'pos_cash_clean_FE.parquet')

os.makedirs(output_dir, exist_ok=True)  # Tự động tạo folder nếu chưa có
pos_agg.to_parquet(output_parquet_path, compression='snappy')  # Ghi file Parquet